In [1]:
import pandas as pd
import numpy as np
import gc
from datetime import datetime
from tqdm import tqdm

In [2]:
# ====================== 配置参数 ======================
INDEX_CODE = "000985.XSHG"
LOOKBACK_DAYS = 100        # 拉取K线数量
MA_WINDOW = 20             # 20日均线（扩散度）
CORR_WINDOW = 60           # 相关系数滚动窗口
SAMPLE_STOCKS = 4881        # 抽样个股数量，400推荐；300更快，500精度更高
MIN_VALID_RATIO = 0.8      # 个股有效数据最低占比，过滤烂数据
date_today = datetime.now()

In [3]:
# 1. 获取中证全指全部成分股
stocks_all = get_index_stocks(INDEX_CODE, date=date_today)
print(f"中证全指原始成分股数量：{len(stocks_all)}")

中证全指原始成分股数量：4881


In [4]:
# 2. 一次性拉全部成分股收盘价
df_price = get_price(
    stocks_all,
    end_date=date_today,
    count=LOOKBACK_DAYS,
    frequency='1d',
    fields=['close'],
    skip_paused=False,
    fq='pre',
    panel=False,
    fill_paused=True
)
print(f"df_price shape: {df_price.shape}")
df_close = df_price.pivot(index="time", columns="code", values="close")
df_ret = df_close.pct_change()
df_ret = df_ret.dropna(how="all")
print(f"收益率矩阵 shape (日期 × 个股): {df_ret.shape}")

df_price shape: (488100, 3)
收益率矩阵 shape (日期 × 个股): (99, 4881)


In [5]:
# ========== 新增：清洗个股，剔除缺失过多标的 ==========
valid_ratio = df_ret.count(axis=0) / len(df_ret)
good_stocks = valid_ratio[valid_ratio >= MIN_VALID_RATIO].index.tolist()
print(f"清洗后，有效股票数量：{len(good_stocks)}")

# 固定种子抽样
rng = np.random.default_rng(seed=42)
stocks_sample = rng.choice(good_stocks, size=SAMPLE_STOCKS, replace=False).tolist()
df_ret_sample = df_ret[stocks_sample].copy()
print(f"抽样后用于相关系数计算的股票数量：{len(stocks_sample)}")

清洗后，有效股票数量：4881
抽样后用于相关系数计算的股票数量：4881


In [6]:
# ====================== 指标1：横截面收益离散度（全部有效个股，不抽样！） ======================
# 横截面离散度，要用全部有效个股，不要抽样，保证指标准确性
cross_section_std = df_ret[good_stocks].std(axis=1)
cross_section_std.name = "横截面离散度"

In [7]:
# ====================== 指标2：市场扩散度：20日均线上个股占比（全部有效个股，不抽样！） ======================
df_close_good = df_close[good_stocks]
df_ma20 = df_close_good.rolling(window=MA_WINDOW).mean()
above_ma20 = (df_close_good > df_ma20)
diffusion_ratio = above_ma20.sum(axis=1) / above_ma20.count(axis=1)
diffusion_ratio.name = "20日均线以上个股占比_扩散度"

In [ ]:
# ====================== 指标3 滚动平均相关系数 ======================
def calc_avg_corr(ret_df: pd.DataFrame):
    market_ret = ret_df.mean(axis=1)
    mask_down = market_ret < 0
    mask_up = market_ret > 0

    # 全部交易日相关
    corr_all = ret_df.corr()
    tri = np.triu(np.ones_like(corr_all, dtype=bool), k=1)
    avg_all = corr_all.where(tri).stack().mean()
    del corr_all

    # 下行相关
    ret_down = ret_df.loc[mask_down]
    if len(ret_down) >= 20:
        corr_down = ret_down.corr()
        tri_d = np.triu(np.ones_like(corr_down, dtype=bool), k=1)
        avg_down = corr_down.where(tri_d).stack().mean()
        del corr_down
    else:
        avg_down = np.nan

    # 上行相关
    ret_up = ret_df.loc[mask_up]
    if len(ret_up) >= 20:
        corr_up = ret_up.corr()
        tri_u = np.triu(np.ones_like(corr_up, dtype=bool), k=1)
        avg_up = corr_up.where(tri_u).stack().mean()
        del corr_up
    else:
        avg_up = np.nan
    gc.collect() # 强制回收内存
    return avg_all, avg_down, avg_up

# 滚动循环 + tqdm进度条
rolling_avg_corr_all = []
rolling_avg_corr_down = []
rolling_avg_corr_up = []
date_list = []

start_idx = CORR_WINDOW
total_iter = len(df_ret_sample) - start_idx

In [ ]:
for i in tqdm(range(start_idx, len(df_ret_sample)), total=total_iter, desc="计算滚动相关系数"):
    try:
        sub_ret = df_ret_sample.iloc[i-CORR_WINDOW:i].copy()
        a_all, a_down, a_up = calc_avg_corr(sub_ret)
        rolling_avg_corr_all.append(a_all)
        rolling_avg_corr_down.append(a_down)
        rolling_avg_corr_up.append(a_up)
        date_list.append(df_ret_sample.index[i])
        del sub_ret
        gc.collect()
    except Exception as e:
        print(f"窗口{i}计算失败，跳过，原因：{str(e)[:200]}")
        rolling_avg_corr_all.append(np.nan)
        rolling_avg_corr_down.append(np.nan)
        rolling_avg_corr_up.append(np.nan)
        date_list.append(df_ret_sample.index[i])
        gc.collect()

In [ ]:
df_corr = pd.DataFrame({
    "滚动60日_全区间平均相关系数": rolling_avg_corr_all,
    "滚动60日_下行平均相关系数": rolling_avg_corr_down,
    "滚动60日_上行平均相关系数": rolling_avg_corr_up
}, index=date_list)

# ====================== 合并模块一所有指标 ======================
df_module1 = pd.concat([
    cross_section_std,
    diffusion_ratio,
], axis=1)
df_module1 = df_module1.join(df_corr, how="outer")

In [ ]:
# 输出最新一行
latest = df_module1.iloc[-1]
print("\n===== 模块一【市场共振风险】最新指标 =====")
print(latest)

# 保存结果
df_module1.to_csv("module1_market_resonance_4881_stock.csv", encoding="utf-8-sig")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# ===================== 绘图 =====================
fig, axes = plt.subplots(3,1, figsize=(16,12), sharex=True)
fig.suptitle("模块一｜市场共振风险指标时序图", fontsize=16, y=0.96)

# 子图1：横截面离散度
ax1 = axes[0]
ax1.plot(df_module1.index, df_module1["横截面离散度"], color="#1f77b4", linewidth=1.5, label="横截面离散度")
ax1.set_title("横截面收益离散度（越低=市场趋同，共振风险上升）")
ax1.legend(loc="upper right")
ax1.set_ylabel("离散度")

# 子图2：市场扩散度，20日均线上个股占比
ax2 = axes[1]
ax2.plot(df_module1.index, df_module1["20日均线以上个股占比_扩散度"], color="#ff7f0e", linewidth=1.5, label="20日均线上个股占比")
ax2.set_title("市场扩散度（越低赚钱效应越弱）")
ax2.legend(loc="upper right")
ax2.set_ylabel("占比")

# 子图3：三组相关系数，重点看【下行平均相关系数】
ax3 = axes[2]
ax3.plot(df_module1.index, df_module1["滚动60日_全区间平均相关系数"], color="#2ca02c", alpha=0.7, label="全区间平均相关")
ax3.plot(df_module1.index, df_module1["滚动60日_上行平均相关系数"], color="#1f77b4", alpha=0.7, label="上行相关")
ax3.plot(df_module1.index, df_module1["滚动60日_下行平均相关系数"], color="#d62728", linewidth=2, label="下行相关【核心预警】")
ax3.set_title("滚动60日个股平均相关系数｜红色=下跌时段共振强度")
ax3.legend(loc="upper right")
ax3.set_ylabel("相关系数")
ax3.set_xlabel("日期")

plt.tight_layout()
plt.show()


In [ ]:
# 计算下行相关系数的历史分位
from scipy import stats

ser_down_corr = df_module1["滚动60日_下行平均相关系数"].dropna()
# 逐行计算分位值
down_corr_pct = ser_down_corr.apply(lambda x: stats.percentileofscore(ser_down_corr, x))

fig, ax = plt.subplots(figsize=(16,5))
ax.plot(ser_down_corr.index, ser_down_corr.values, color="red", lw=1.5, label="下行相关系数")
ax2 = ax.twinx()
ax2.plot(down_corr_pct.index, down_corr_pct.values, color="black", lw=1, linestyle="--", label="历史分位(%)")
ax.set_title("滚动60日下行相关系数 + 历史分位", fontsize=14)
ax.set_ylabel("相关系数", color="red")
ax2.set_ylabel("历史分位 %", color="black")
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16,6))
ax.plot(df_module1.index, df_module1["横截面离散度"], label="横截面离散度", lw=1.2)
ax.plot(df_module1.index, df_module1["20日均线以上个股占比_扩散度"], label="20日线上占比", lw=1.2)
ax.plot(df_module1.index, df_module1["滚动60日_下行平均相关系数"], label="下行相关系数", lw=1.5, c='r')
ax.set_title("模块一三大共振指标合并时序", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()